# The Normal Equation for Linear Regression

This notebook explores the **normal equation**, a closed-form (non-iterative) solution for finding optimal parameters in linear regression. Unlike gradient descent which requires iterations, the normal equation directly computes the best parameters in one step by solving a system of linear equations.

## Learning Objectives
By the end of this notebook, you will be able to:
1. **Derive** the Normal Equation $\theta^* = (X^TX)^{-1}X^Ty$ by setting $\nabla_\theta J(\theta) = 0$ and solving the resulting system.
2. **Explain** when $X^TX$ is invertible (full column rank) and what multicollinearity means for the solution.
3. **Compare** four numerical methods (direct inversion, linear system solve, QR, SVD) by stability and cost.
4. **Apply** Ridge regression $\theta^* = (X^TX + \lambda I)^{-1}X^Ty$ to handle ill-conditioned or singular matrices.
5. **Benchmark** Normal Equation vs gradient descent and choose the appropriate method based on $n$ and $p$.
6. **Use** `np.linalg.solve` and `sklearn.linear_model.LinearRegression` to compute the Normal Equation solution.

> **Prerequisite**: Linear algebra ([ml_000_00](ml_000_00_linear_algebra.ipynb)), batch gradient descent ([ml_001_02](ml_001_02_batch_gradient_descent.ipynb)).

## Important Note on Mathematical Notation

Throughout this notebook, whenever we use a mathematical concept or formula, we first present the **general foundational principle** before applying it to our specific problem. This approach ensures clarity by:

1. Introducing the general concept and formula
2. Explaining what each component means
3. Showing how we apply it to the normal equation

This makes it easier to understand not just *how* the normal equation works, but *why* it works and where it comes from.

## 1. Introduction to the Normal Equation

### 1.1 The Problem We're Solving

In linear regression, we want to find parameters $\theta$ that minimize the Mean Squared Error (MSE):

$$J(\theta) = \frac{1}{n}\|y - X\theta\|_2^2 = \frac{1}{n}(y - X\theta)^T(y - X\theta)$$

where:
- $X \in \mathbb{R}^{n \times p}$: Feature matrix (rows are samples, columns are features)
- $y \in \mathbb{R}^{n}$: Target vector
- $\theta \in \mathbb{R}^{p}$: Parameter vector (what we want to find)
- $n$: Number of samples
- $p$: Number of features

### 1.2 Two Approaches to Find Optimal $\theta$

**Iterative Approach (Gradient Descent)**

Start with a guess $\theta^{(0)}$ and iteratively improve it:

$$\theta^{(t+1)} = \theta^{(t)} - \alpha \nabla J(\theta^{(t)})$$

Pros: Works for all problem sizes, works with non-convex functions

Cons: Requires tuning learning rate, convergence is iterative

**Direct/Closed-Form Approach (Normal Equation)**

Directly solve for $\theta$ by setting gradient to zero:

$$\theta^* = (X^T X)^{-1} X^T y$$

Pros: One-step solution, no hyperparameters to tune

Cons: Expensive for large $n$ and $p$, requires matrix inversion

### 1.3 When to Use the Normal Equation

| Criterion | Use Normal Equation | Use Gradient Descent |
|-----------|-------------------|---------------------|
| Dataset size | Small to medium ($n < 10,000$) | Large ($n > 10,000$) |
| Number of features | Small to medium ($p < 1,000$) | Large ($p > 1,000$) |
| Interpretability | Want exact solution quickly | Learning process valuable |
| Online learning | No (need all data at once) | Yes (one sample at a time) |

## 2. Mathematical Derivation of the Normal Equation

### 2.1 General Principle: Minimizing Convex Functions

**General Principle**: For a convex function $f(x)$, the minimum occurs where the gradient equals zero:

$$\nabla f(x^*) = 0 \implies x^* \text{ is optimal}$$

This is a first-order necessary condition. For strictly convex functions (like our MSE with $X^T X$ invertible), it's also sufficient.

### 2.2 Applying to MSE Loss

Our loss function is:

$$J(\theta) = \frac{1}{n}(y - X\theta)^T(y - X\theta)$$

**Step 1: Expand the squared norm**

$$J(\theta) = \frac{1}{n}[(y - X\theta)^T(y - X\theta)]$$
$$= \frac{1}{n}[y^T y - y^T X\theta - \theta^T X^T y + \theta^T X^T X\theta]$$

Note: $y^T X\theta$ and $\theta^T X^T y$ are scalars, so they're equal: $y^T X\theta = \theta^T X^T y$

$$= \frac{1}{n}[y^T y - 2\theta^T X^T y + \theta^T X^T X\theta]$$

**Step 2: Take the gradient with respect to $\theta$**

We know from calculus:
- $\nabla_\theta (\theta^T a) = a$ (where $a$ is constant)
- $\nabla_\theta (\theta^T A\theta) = 2A\theta$ (where $A$ is symmetric)
- $\nabla_\theta (\text{const}) = 0$

Therefore:

$$\nabla_\theta J(\theta) = \frac{1}{n}[0 - 2X^T y + 2X^T X\theta]$$
$$= \frac{2}{n}[X^T X\theta - X^T y]$$

**Step 3: Set gradient to zero**

$$\nabla_\theta J(\theta) = 0$$
$$\frac{2}{n}[X^T X\theta - X^T y] = 0$$
$$X^T X\theta = X^T y$$

**Step 4: Solve for $\theta$**

If $X^T X$ is invertible (i.e., $X$ has full column rank):

$$\theta^* = (X^T X)^{-1} X^T y$$

This is the **normal equation**!

### 2.3 Interpretation of the Normal Equation

The equation $X^T X\theta = X^T y$ has a beautiful interpretation:

- $X^T X$: Information matrix (how much information each feature has)
- $X^T y$: Correlation between features and target
- $(X^T X)^{-1}$: Precision matrix (inverse of information)

So $\theta^* = (X^T X)^{-1} X^T y$ means: **weight the correlations by the inverse information**

## 3. The Pseudoinverse and Full Picture

### 3.1 The Pseudoinverse

The normal equation can be written as:

$$\theta^* = (X^T X)^{-1} X^T y = X^\dagger y$$

where $X^\dagger = (X^T X)^{-1} X^T$ is called the **Moore-Penrose pseudoinverse** of $X$.

**Properties**:
- $X^\dagger$ has shape $(p, n)$ (transpose of $X$'s shape)
- Even if $X$ is not square and not invertible, $X^\dagger$ exists
- $(X^\dagger)^\dagger = X$
- $X X^\dagger X = X$

### 3.2 When is $X^T X$ Invertible?

$X^T X$ is invertible if and only if **$X$ has full column rank**, meaning:
- No column of $X$ is a linear combination of other columns
- All $p$ features are linearly independent
- $n \geq p$ (at least as many samples as features)
- $\text{rank}(X) = p$

**If $X^T X$ is singular** (not invertible):
- Features are linearly dependent (multicollinearity)
- Infinitely many optimal solutions exist
- Use regularization (Ridge regression) or remove redundant features

## 4. Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

np.random.seed(42)

### 4.1 Generate Synthetic Data

In [ ]:
# Generate synthetic linear regression data
n_samples = 100
n_features = 2

# True parameters (what we're trying to recover)
# Intercept=2, Slope=3
true_theta = np.array([[2.0], [3.0]])
# Noise level
true_sigma = 0.5

# Generate input features
X_base = np.random.randn(n_samples, n_features - 1)
# Add bias column (ones)
X = np.hstack([np.ones((n_samples, 1)), X_base])

# Generate targets with noise
y = X @ true_theta + np.random.randn(n_samples, 1) * true_sigma

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFirst 5 samples:")
print(f"X[0:5] =\n{X[0:5]}")
print(f"y[0:5] = {y[0:5].flatten()}")

# Visualize
plt.figure(figsize=(10, 5))
plt.scatter(X[:, 1], y, alpha=0.6, s=50)
plt.xlabel('Feature x')
plt.ylabel('Target y')
plt.title('Synthetic Linear Regression Data')
plt.grid(True, alpha=0.3)
plt.show()


### 4.2 Normal Equation Implementation

In [ ]:
def normal_equation(X, y):
    """
    Solve linear regression using the normal equation.
    
    θ* = (X^T X)^(-1) X^T y
    
    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target vector (n_samples, 1)
    
    Returns:
        theta: Optimal parameters (n_features, 1)
        XTX: X^T X matrix (useful for analysis)
        XTy: X^T y vector (useful for analysis)
    """
    # Compute X^T X and X^T y
    # (p, n) @ (n, p) = (p, p)
    XTX = X.T @ X
    # (p, n) @ (n, 1) = (p, 1)
    XTy = X.T @ y
    
    # Solve the system: (X^T X) θ = X^T y
    # Method 1: Using matrix inversion
    try:
        theta = np.linalg.inv(XTX) @ XTy
        return theta, XTX, XTy
    except np.linalg.LinAlgError:
        print("Warning: X^T X is singular. Using pseudoinverse instead.")
        theta = np.linalg.pinv(XTX) @ XTy
        return theta, XTX, XTy

# Solve using normal equation
print("Solving with Normal Equation...")
theta_ne, XTX, XTy = normal_equation(X, y)

print(f"\nLearned parameters: {theta_ne.flatten()}")
print(f"True parameters: {true_theta.flatten()}")
print(f"Parameter error: {np.linalg.norm(theta_ne - true_theta):.6f}")

# Check matrix properties
print(f"\nX^T X shape: {XTX.shape}")
print(f"X^T X =\n{XTX}")
print(f"\nCondition number of X^T X: {np.linalg.cond(XTX):.4f}")
print(f"(Small is good; values > 1e10 indicate numerical problems)")

# Check if solution minimizes loss
y_pred = X @ theta_ne
mse = np.mean((y - y_pred)**2)
print(f"\nMean Squared Error: {mse:.6f}")


### 4.3 Verification: Is This Really the Minimum?

In [ ]:
# Compute loss at the normal equation solution
def compute_loss(X, y, theta):
    y_pred = X @ theta
    return np.mean((y - y_pred)**2) / 2

# Verify by checking gradient is near zero
def compute_gradient(X, y, theta):
    n = X.shape[0]
    y_pred = X @ theta
    error = y_pred - y
    grad = (1.0 / n) * (X.T @ error)
    return grad

# Check at normal equation solution
grad_ne = compute_gradient(X, y, theta_ne)
loss_ne = compute_loss(X, y, theta_ne)

print("At Normal Equation Solution:")
print(f"Gradient: {grad_ne.flatten()}")
print(f"Gradient magnitude: {np.linalg.norm(grad_ne):.2e}")
print(f"Loss: {loss_ne:.6f}")
print(f"\n(Gradient near zero ✓ means we're at a critical point)")

# Verify by checking loss in a neighborhood
perturbations = np.linspace(-0.5, 0.5, 100)
losses = []

for pert in perturbations:
    theta_perturbed = theta_ne + pert * np.array([[0.1], [0.1]])
    loss = compute_loss(X, y, theta_perturbed)
    losses.append(loss)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(perturbations, losses, 'b-', linewidth=2)
plt.scatter([0], [loss_ne], color='r', s=100, zorder=5, label='Normal Equation Solution')
plt.xlabel('Perturbation')
plt.ylabel('Loss')
plt.title('Loss Surface Near Solution')
plt.legend()
plt.grid(True, alpha=0.3)

# 2D contour plot of loss function
theta0_range = np.linspace(0.5, 3.5, 50)
theta1_range = np.linspace(1.0, 5.0, 50)
T0, T1 = np.meshgrid(theta0_range, theta1_range)
Z = np.zeros_like(T0)

for i in range(len(theta0_range)):
    for j in range(len(theta1_range)):
        theta_temp = np.array([[T0[j, i]], [T1[j, i]]])
        Z[j, i] = compute_loss(X, y, theta_temp)

plt.subplot(1, 2, 2)
contour = plt.contour(T0, T1, Z, levels=15, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)
plt.scatter(theta_ne[0], theta_ne[1], color='r', s=200, marker='*', 
           label='Normal Eq.', zorder=5, edgecolors='black', linewidth=1)
plt.scatter(true_theta[0], true_theta[1], color='g', s=200, marker='^', 
           label='True θ', zorder=5, edgecolors='black', linewidth=1)
plt.xlabel('θ₀')
plt.ylabel('θ₁')
plt.title('Loss Contour Plot')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Minimum loss at Normal Equation: {loss_ne:.6f}")

## 5. Alternative Numerical Methods

### 5.1 Method 1: Direct Inversion (Standard)

$\theta = (X^T X)^{-1} X^T y$

Advantages: Clear mathematical form

Disadvantages: Numerically unstable, $O(p^3)$ time

### 5.2 Method 2: Solve Linear System (Better)

Solve $(X^T X)\theta = X^T y$ directly using `np.linalg.solve()`

Advantages: More numerically stable, $O(p^3)$ but better constants

Disadvantages: Requires invertible matrix

### 5.3 Method 3: QR Decomposition (Best)

Decompose $X = QR$ where $Q$ is orthogonal, $R$ is upper triangular

Then: $\theta = R^{-1} Q^T y$

Advantages: Very numerically stable, $O(np^2)$

Disadvantages: Slightly more complex

### 5.4 Method 4: Singular Value Decomposition (Most Robust)

Decompose $X = U\Sigma V^T$

Then: $\theta = V\Sigma^{-1}U^T y$ (uses pseudoinverse)

Advantages: Handles rank-deficient matrices, very stable

Disadvantages: $O(np^2)$, more expensive

In [ ]:
# Compare different numerical methods
print("Method Comparison:")
print("="*70)

# Method 1: Direct inversion
start = time.time()
theta_inv = np.linalg.inv(X.T @ X) @ X.T @ y
time_inv = time.time() - start
loss_inv = compute_loss(X, y, theta_inv)
print(f"\n1. Direct Inversion: θ = (X^T X)^(-1) X^T y")
print(f"   Time: {time_inv*1e6:.2f} μs")
print(f"   Loss: {loss_inv:.6f}")
print(f"   Parameters: {theta_inv.flatten()}")

# Method 2: Solve linear system
start = time.time()
theta_solve = np.linalg.solve(X.T @ X, X.T @ y)
time_solve = time.time() - start
loss_solve = compute_loss(X, y, theta_solve)
print(f"\n2. Solve Linear System: (X^T X)θ = X^T y")
print(f"   Time: {time_solve*1e6:.2f} μs")
print(f"   Loss: {loss_solve:.6f}")
print(f"   Parameters: {theta_solve.flatten()}")

# Method 3: QR decomposition
start = time.time()
Q, R = np.linalg.qr(X)
theta_qr = np.linalg.solve(R, Q.T @ y)
time_qr = time.time() - start
loss_qr = compute_loss(X, y, theta_qr)
print(f"\n3. QR Decomposition: X = QR, θ = R^(-1) Q^T y")
print(f"   Time: {time_qr*1e6:.2f} μs")
print(f"   Loss: {loss_qr:.6f}")
print(f"   Parameters: {theta_qr.flatten()}")

# Method 4: SVD (Pseudoinverse)
start = time.time()
U, s, Vt = np.linalg.svd(X, full_matrices=False)
theta_svd = Vt.T @ np.diag(1/s) @ U.T @ y
time_svd = time.time() - start
loss_svd = compute_loss(X, y, theta_svd)
print(f"\n4. Singular Value Decomposition: X = UΣV^T, θ = VΣ^(-1)U^T y")
print(f"   Time: {time_svd*1e6:.2f} μs")
print(f"   Loss: {loss_svd:.6f}")
print(f"   Parameters: {theta_svd.flatten()}")

# Method 5: NumPy's lstsq (least squares)
start = time.time()
theta_lstsq, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
time_lstsq = time.time() - start
loss_lstsq = compute_loss(X, y, theta_lstsq)
print(f"\n5. NumPy lstsq: Optimized least squares solver")
print(f"   Time: {time_lstsq*1e6:.2f} μs")
print(f"   Loss: {loss_lstsq:.6f}")
print(f"   Parameters: {theta_lstsq.flatten()}")

print(f"\n" + "="*70)
print(f"All methods give the same solution (differences are numerical errors)")

## 6. Computational Complexity Analysis

In [ ]:
# Analyze computational complexity
print("Computational Complexity Analysis")
print("="*70)

# Create datasets of increasing size
sizes = [(100, 5), (500, 10), (1000, 20), (5000, 50), (10000, 100)]

times_inv = []
times_solve = []
times_qr = []
times_svd = []
times_lstsq = []

for n, p in sizes:
    # Generate data
    X_temp = np.hstack([np.ones((n, 1)), np.random.randn(n, p-1)])
    y_temp = np.random.randn(n, 1)
    
    # Time each method
    n_runs = 10
    
    start = time.time()
    for _ in range(n_runs):
        _ = np.linalg.inv(X_temp.T @ X_temp) @ X_temp.T @ y_temp
    times_inv.append((time.time() - start) / n_runs)
    
    start = time.time()
    for _ in range(n_runs):
        _ = np.linalg.solve(X_temp.T @ X_temp, X_temp.T @ y_temp)
    times_solve.append((time.time() - start) / n_runs)
    
    start = time.time()
    for _ in range(n_runs):
        Q, R = np.linalg.qr(X_temp)
        _ = np.linalg.solve(R, Q.T @ y_temp)
    times_qr.append((time.time() - start) / n_runs)
    
    start = time.time()
    for _ in range(n_runs):
        U, s, Vt = np.linalg.svd(X_temp, full_matrices=False)
        _ = Vt.T @ np.diag(1/s) @ U.T @ y_temp
    times_svd.append((time.time() - start) / n_runs)
    
    start = time.time()
    for _ in range(n_runs):
        _, _, _, _ = np.linalg.lstsq(X_temp, y_temp, rcond=None)
    times_lstsq.append((time.time() - start) / n_runs)

print(f"\nDataset Size (n, p) | Inv (ms) | Solve (ms) | QR (ms) | SVD (ms) | lstsq (ms)")
print("-" * 80)
for (n, p), t_inv, t_solve, t_qr, t_svd, t_lstsq in zip(
    sizes, times_inv, times_solve, times_qr, times_svd, times_lstsq
):
    print(f"({n:5d}, {p:3d})        | {t_inv*1e3:7.3f}  | {t_solve*1e3:9.3f}  | {t_qr*1e3:7.3f} | {t_svd*1e3:7.3f}  | {t_lstsq*1e3:8.3f}")

print(f"\n" + "="*70)
print(f"\nComplexity Summary:")
print(f"Method          | Time Complexity | Space Complexity")
print(f"-" * 60)
print(f"Direct Inv      | O(p³)          | O(p²)")
print(f"Solve           | O(p³)          | O(p²)")
print(f"QR              | O(np²)         | O(np)")
print(f"SVD             | O(np²)         | O(np)")
print(f"lstsq (auto)    | Varies         | Varies")
print(f"\nNote: For n >> p, QR and SVD are faster. Direct methods are O(p³) regardless of n.")

## 7. Normal Equation vs. Gradient Descent

In [ ]:
# Implement simple gradient descent for comparison
def batch_gradient_descent(X, y, learning_rate=0.01, n_iterations=1000):
    theta = np.zeros((X.shape[1], 1))
    loss_history = []
    
    for iteration in range(n_iterations):
        y_pred = X @ theta
        loss = np.mean((y - y_pred)**2) / 2
        loss_history.append(loss)
        
        error = y_pred - y
        grad = (1.0 / X.shape[0]) * (X.T @ error)
        theta = theta - learning_rate * grad
    
    return theta, loss_history

# Run GD
print("Comparing Normal Equation vs. Gradient Descent...\n")

# Regenerate data for fair comparison
n_samples, n_features = 100, 2
true_theta_new = np.array([[2.0], [3.0]])
X_new = np.hstack([np.ones((n_samples, 1)), np.random.randn(n_samples, n_features-1)])
y_new = X_new @ true_theta_new + np.random.randn(n_samples, 1) * 0.5

# Normal Equation
start = time.time()
theta_ne_new, _, _ = normal_equation(X_new, y_new)
time_ne = time.time() - start

# Gradient Descent
start = time.time()
theta_gd, loss_history = batch_gradient_descent(X_new, y_new, learning_rate=0.1, n_iterations=1000)
time_gd = time.time() - start

print(f"Normal Equation:")
print(f"  Time: {time_ne*1e6:.2f} μs")
print(f"  Parameters: {theta_ne_new.flatten()}")
print(f"  Loss: {compute_loss(X_new, y_new, theta_ne_new):.8f}")

print(f"\nGradient Descent (1000 iterations):")
print(f"  Time: {time_gd*1e3:.2f} ms")
print(f"  Parameters: {theta_gd.flatten()}")
print(f"  Loss: {loss_history[-1]:.8f}")

print(f"\nSpeedup: Normal Equation is {time_gd/time_ne:.0f}x faster!")

# Visualization
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_history, 'b-', linewidth=2, label='GD (1000 iterations)')
plt.axhline(y=compute_loss(X_new, y_new, theta_ne_new), color='r', linestyle='--', 
           linewidth=2, label='Normal Equation')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss Convergence: GD vs. Normal Equation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.subplot(1, 2, 2)
x_vals = np.linspace(-3, 3, 100).reshape(-1, 1)
X_plot = np.hstack([np.ones_like(x_vals), x_vals])
y_ne = X_plot @ theta_ne_new
y_gd = X_plot @ theta_gd

plt.scatter(X_new[:, 1], y_new, alpha=0.5, s=30, label='Data')
plt.plot(x_vals, y_ne, 'r-', linewidth=2.5, label='Normal Equation')
plt.plot(x_vals, y_gd, 'b--', linewidth=2, alpha=0.7, label='Gradient Descent')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Fitted Models')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Regularized Normal Equation: Ridge Regression

### 8.1 The Problem: Multicollinearity

When features are highly correlated (multicollinearity), $X^T X$ becomes ill-conditioned or singular:
- Condition number becomes very large
- Numerical instability
- Solution becomes unreliable

### 8.2 Ridge Regression Solution

Add L2 regularization term to prevent overfitting:

$$J(\theta) = \frac{1}{n}\|y - X\theta\|_2^2 + \lambda\|\theta\|_2^2$$

where $\lambda \geq 0$ is the regularization parameter.

The closed-form solution becomes:

$$\theta^* = (X^T X + \lambda I)^{-1} X^T y$$

where $I$ is the identity matrix.

**Why it helps:**
- Adding $\lambda I$ makes the matrix better conditioned (less singular)
- $\lambda$ controls trade-off between fitting data vs. keeping parameters small
- $\lambda = 0$ gives normal equation; $\lambda \to \infty$ gives $\theta \to 0$

In [ ]:
def ridge_regression(X, y, lambda_param=0.1):
    """
    Ridge regression (L2 regularized normal equation).
    
    θ* = (X^T X + λI)^(-1) X^T y
    """
    p = X.shape[1]
    I = np.eye(p)
    
    # Regularized normal equation
    XTX_reg = X.T @ X + lambda_param * I
    XTy = X.T @ y
    
    theta = np.linalg.solve(XTX_reg, XTy)
    return theta, XTX_reg

# Compare different regularization strengths
lambda_values = [0, 0.01, 0.1, 1.0, 10.0]
thetas_ridge = {}
cond_numbers = {}

print("Ridge Regression: Effect of Regularization Parameter λ\n")
print(f"λ      | Condition # | θ₀        | θ₁        | ||θ||    | Loss")
print("-" * 75)

for lam in lambda_values:
    theta, XTX_reg = ridge_regression(X_new, y_new, lambda_param=lam)
    thetas_ridge[lam] = theta
    cond_numbers[lam] = np.linalg.cond(XTX_reg)
    
    theta_norm = np.linalg.norm(theta)
    loss = compute_loss(X_new, y_new, theta)
    
    print(f"{lam:5.2f} | {cond_numbers[lam]:11.2f} | {theta[0, 0]:9.4f} | {theta[1, 0]:9.4f} | {theta_norm:9.4f} | {loss:.6f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss vs lambda
ax = axes[0, 0]
lambdas_fine = np.logspace(-3, 2, 50)
losses = []
for lam in lambdas_fine:
    theta, _ = ridge_regression(X_new, y_new, lambda_param=lam)
    losses.append(compute_loss(X_new, y_new, theta))
ax.semilogx(lambdas_fine, losses, 'b-', linewidth=2)
for lam in lambda_values:
    theta, _ = ridge_regression(X_new, y_new, lambda_param=lam)
    ax.scatter(lam, compute_loss(X_new, y_new, theta), s=100, zorder=5)
ax.set_xlabel('λ (regularization strength)')
ax.set_ylabel('Loss on training data')
ax.set_title('Training Loss vs. Regularization')
ax.grid(True, alpha=0.3)

# Parameter norm vs lambda
ax = axes[0, 1]
param_norms = []
for lam in lambdas_fine:
    theta, _ = ridge_regression(X_new, y_new, lambda_param=lam)
    param_norms.append(np.linalg.norm(theta))
ax.loglog(lambdas_fine, param_norms, 'r-', linewidth=2)
for lam in lambda_values:
    theta, _ = ridge_regression(X_new, y_new, lambda_param=lam)
    ax.scatter(lam, np.linalg.norm(theta), s=100, zorder=5)
ax.set_xlabel('λ')
ax.set_ylabel('||θ||')
ax.set_title('Parameter Norm vs. Regularization')
ax.grid(True, alpha=0.3, which='both')

# Fitted lines
ax = axes[1, 0]
x_vals = np.linspace(-3, 3, 100).reshape(-1, 1)
X_plot = np.hstack([np.ones_like(x_vals), x_vals])
colors_ridge = plt.cm.viridis(np.linspace(0, 1, len(lambda_values)))
ax.scatter(X_new[:, 1], y_new, alpha=0.3, s=30, color='black', label='Data')
for lam, color in zip(lambda_values, colors_ridge):
    y_pred = X_plot @ thetas_ridge[lam]
    ax.plot(x_vals, y_pred, linewidth=2.5, label=f'λ={lam}', color=color)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Ridge Regression: Different Regularization Strengths')
ax.legend()
ax.grid(True, alpha=0.3)

# Condition number vs lambda
ax = axes[1, 1]
cond_vals = [cond_numbers[lam] for lam in lambdas_fine]
ax.loglog(lambdas_fine, cond_vals, 'g-', linewidth=2)
for lam in lambda_values:
    ax.scatter(lam, cond_numbers[lam], s=100, zorder=5)
ax.set_xlabel('λ')
ax.set_ylabel('Condition Number')
ax.set_title('Numerical Stability vs. Regularization')
ax.grid(True, alpha=0.3, which='both')
ax.axhline(y=1e10, color='r', linestyle='--', alpha=0.5, label='Numerical Issues')
ax.legend()

plt.tight_layout()
plt.show()

## 9. Library Functions for Normal Equation

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

print("Using Scikit-Learn for Normal Equation\n")
print("="*70)

# Standard Linear Regression (uses normal equation internally)
lr = LinearRegression()
lr.fit(X_new, y_new.flatten())

print(f"\n1. LinearRegression (uses normal equation)")
print(f"   Intercept: {lr.intercept_:.4f}")
print(f"   Coefficients: {lr.coef_}")
print(f"   R² score: {lr.score(X_new, y_new.flatten()):.6f}")

# Ridge Regression
# Note: sklearn uses alpha instead of lambda
ridge = Ridge(alpha=0.1)
ridge.fit(X_new, y_new.flatten())

print(f"\n2. Ridge Regression (regularized normal equation)")
print(f"   Intercept: {ridge.intercept_:.4f}")
print(f"   Coefficients: {ridge.coef_}")
print(f"   R² score: {ridge.score(X_new, y_new.flatten()):.6f}")

# Predictions
y_pred_lr = lr.predict(X_new)
y_pred_ridge = ridge.predict(X_new)

print(f"\n" + "="*70)
print(f"\nCode Examples:")
print(f"-"*70)

code_example = """
from sklearn.linear_model import LinearRegression, Ridge
import numpy as np

# Linear Regression using normal equation
lr = LinearRegression()
lr.fit(X, y)
theta = np.hstack([lr.intercept_, lr.coef_])
y_pred = lr.predict(X_new)

# Ridge Regression (regularized)
ridge = Ridge(alpha=0.1)  # Regularization strength
ridge.fit(X, y)
y_pred_ridge = ridge.predict(X_new)

# Get R² score (coefficient of determination)
r2 = lr.score(X, y)
"""

print(code_example)


## 10. Advantages and Disadvantages

### 10.1 Normal Equation Advantages

✓ **One-step solution**: No iterations, no hyperparameters to tune

✓ **Exact answer**: Finds global optimum directly (for well-posed problems)

✓ **No learning rate**: No need to choose step size

✓ **Simple**: Easy to understand and implement

✓ **Interpretable**: Solution shows which features matter

### 10.2 Normal Equation Disadvantages

✗ **Expensive for large $p$**: Requires $O(p^3)$ operations (matrix inversion)

✗ **Memory intensive**: Need to store $X^T X$ (size $p \times p$)

✗ **Singular matrix**: Fails if features are linearly dependent (multicollinearity)

✗ **No online learning**: Must have all data at once

✗ **Not scalable**: Impractical when $p > 10,000$

✗ **No regularization** (standard): Prone to overfitting on high-dimensional data

### 10.3 Comparison Table

| Aspect | Normal Equation | Gradient Descent |
|--------|---|---|
| **Time per solution** | O(p³) or O(np²) | O(Tnp) for T iterations |
| **Hyperparameters** | None | Learning rate α, # iterations |
| **Online learning** | No | Yes (SGD) |
| **Scalability** | Small to medium | Large datasets |
| **Numerical stability** | Can be poor (singular matrices) | Generally stable |
| **Implementation** | Simple | More complex |
| **Parallelization** | Difficult | Easy |

## 11. When to Use Normal Equation vs. Gradient Descent

### Decision Guide

```
Linear Regression Problem
    |
    ├─ Dataset size small (n < 10,000)?
    │  ├─ YES → Number of features small (p < 1,000)?
    │  │       ├─ YES → Use NORMAL EQUATION ✓
    │  │       └─ NO → Use GRADIENT DESCENT
    │  └─ NO → Use GRADIENT DESCENT or STOCHASTIC GD
    │
    ├─ Need online learning (data arrives over time)?
    │  ├─ YES → Use STOCHASTIC GRADIENT DESCENT
    │  └─ NO → Check dataset size above
    │
    └─ Have multicollinearity (correlated features)?
       ├─ YES → Use RIDGE REGRESSION (regularized normal eq.)
       └─ NO → Use NORMAL EQUATION or GD
```

### Practical Recommendations

- **$n = 100$, $p = 5$**: Normal Equation (instant)
- **$n = 1,000$, $p = 50$**: Normal Equation (very fast)
- **$n = 10,000$, $p = 100$**: Either method (normal eq ~1 sec, GD ~0.1 sec)
- **$n = 100,000$, $p = 1,000$**: Gradient Descent (normal eq infeasible)
- **$n = 1,000,000$, $p = 10,000$**: Stochastic GD or Mini-batch GD
- **Streaming/Online**: Stochastic GD with `partial_fit()`

## 12. Key Takeaways

1. **Closed-form Solution**: The normal equation $\theta^* = (X^T X)^{-1} X^T y$ solves linear regression in one step

2. **Derivation**: Comes from setting $\nabla_\theta J(\theta) = 0$ and solving the resulting linear system

3. **Three Approaches**:
    - Direct inversion: $\theta = (X^T X)^{-1} X^T y$ (simple, less stable)
    - Solve system: $(X^T X)\theta = X^T y$ (more stable)
    - Decomposition: QR or SVD (most stable)

4. **Complexity**: $O(p^3)$ for matrix operations (independent of $n$)

5. **When to Use**: Small-medium datasets with $p < 1,000$ features

6. **When NOT to Use**: Large $p$ (high-dimensional), online learning, or multicollinearity

7. **Regularization**: Ridge regression adds $\lambda I$ to fix multicollinearity: $\theta^* = (X^T X + \lambda I)^{-1} X^T y$

8. **Numerical Stability**: Check condition number of $X^T X$ to assess stability

9. **Library Support**: Scikit-learn's `LinearRegression` uses normal equation internally

10. **Trade-off**: Normal equation is exact but expensive; gradient descent is iterative but scalable

## References

- Boyd, S., & Vandenberghe, L. (2004). Convex Optimization. Cambridge University Press.
- Strang, G. (1986). Introduction to Applied Mathematics. Wellesley-Cambridge Press.
- Golub, G. H., & Van Loan, C. F. (1996). Matrix Computations. Johns Hopkins University Press.

---
## Practice Exercises

**Conceptual**

1. Derive the Normal Equation by expanding $J(\theta) = \frac{1}{n}(y - X\theta)^T(y - X\theta)$, differentiating with respect to $\theta$, and setting equal to zero. State which matrix calculus rules you use.

2. What does it mean for $X^TX$ to be singular? Give two data scenarios that cause singularity (other than $n < p$).

3. Ridge regression adds $\lambda I$ to $X^TX$ before inverting. Why does this always make the matrix invertible (for $\lambda > 0$)? What is the effect on the estimated $\theta$ as $\lambda \to \infty$?

4. The QR decomposition method is preferred over direct inversion for numerical stability. Explain why in terms of the condition number of $X^TX$ vs that of $X$ alone.

5. For linear regression, Newton's method converges in exactly one step regardless of initialization. Explain why, using the fact that the loss is quadratic.

**Numerical**

6. Generate a $100 \times 3$ design matrix $X$ (with intercept column) and compute $\hat\theta$ using: (a) direct inversion, (b) `np.linalg.solve`, (c) QR decomposition. Verify all three give the same answer.

7. Create a dataset with two perfectly correlated features ($x_2 = 3x_1$). Attempt the Normal Equation and observe what happens. Then add Ridge regularization $\lambda = 0.1$ and compare the results.

8. Benchmark the Normal Equation against BGD (1000 iterations) for $n=500$, $p \in \{5, 10, 50, 100\}$. At what $p$ does BGD become faster?

**Reflection**

9. A data scientist says "I always use the Normal Equation — it's exact and requires no tuning." In what situation would you push back on this advice?